# 05 · M5 — Pasajeros por segmentación de instancias (YOLO11-seg)
**IAM-811** · Laboratorio Final de Hito · *punto extra*

Entrena un modelo propio de **segmentación de instancias** que segmenta **personas** (y, si el dataset los trae,
vehículos). El servidor cuenta como pasajeros las personas cuya caja está mayoritariamente dentro del vehículo
principal detectado por M2 (≥ 60 %; en moto ≥ 30 % porque el conductor sobresale).

**Privacidad:** no hay reconocimiento facial ni identificación de personas; no se guardan recortes de rostros.
El servidor solo devuelve el número (`pasajeros`) y cajas para dibujar; la web guarda únicamente `pasos.pasajeros`.

**Contrato con `/inferencia`** (lo comprueba la sección 6 antes de exportar):
- archivo final **`pasajeros_seg.pt`** → `inferencia/pesos/pasajeros_seg.pt` (en el Pod: `/workspace/peaje/inferencia/pesos/`)
- `task = segment` y **al menos una clase de persona** (`person`, `persona`, `pasajero`, `driver`, `rider`…).
  Las demás clases se ignoran al contar. Si el archivo no está, el servidor usa `yolo11s-seg.pt` (COCO person).

**GPU T4** · ~30–45 min para 60 épocas con ~1 000 imágenes.

In [ ]:
!nvidia-smi -L || echo "Sin GPU: Entorno de ejecución > Cambiar tipo de entorno > GPU T4"
!pip -q install -U ultralytics==8.4.218 roboflow==1.7.3

## 1. Dataset
**Opción A — Roboflow Universe (recomendada):** un dataset público de *Instance Segmentation* con clase `person` y
vehículos. Valor sugerido: `work-7oioc / vehicle-gtmvy`, versión 2. **Ábrelo en Roboflow Universe y confirma** que sea de
segmentación, que tenga una clase de persona y el número de versión; si no, haz *Fork* de otro y cambia los 3 valores.
La API key se lee de los **Secretos de Colab** (icono de llave, nombre `ROBOFLOW_API_KEY`, con acceso al notebook activado);
nunca se escribe en el notebook.

**Opción B — zip:** un dataset propio exportado como *YOLOv11* (Instance Segmentation).

In [ ]:
import os, glob, yaml, shutil
FUENTE = 'roboflow'          # 'roboflow' o 'zip'
WORKSPACE, PROYECTO, VERSION = 'work-7oioc', 'vehicle-gtmvy', 2
NOMBRE_FINAL = 'pasajeros_seg.pt'     # nombre EXACTO que busca inferencia/app/config.py

def preparar_yaml(ruta_yaml):
    """Deja el data.yaml con rutas absolutas validas en Colab (Roboflow exporta '../train/images', 'valid/', etc.)."""
    d = yaml.safe_load(open(ruta_yaml))
    base = os.path.dirname(os.path.abspath(ruta_yaml))
    d['path'] = base
    for clave, candidatas in (('train', ['train/images', 'images/train']),
                              ('val', ['valid/images', 'val/images', 'images/val']),
                              ('test', ['test/images', 'images/test'])):
        encontrada = next((c for c in candidatas if os.path.isdir(os.path.join(base, c))), None)
        if encontrada:
            d[clave] = encontrada
        elif clave == 'test':
            d.pop('test', None)
        else:
            raise FileNotFoundError(f'No hay carpeta {clave} en {base}. En Roboflow genera la version con split train/valid.')
    yaml.safe_dump(d, open(ruta_yaml, 'w'), allow_unicode=True, sort_keys=False)
    return d

if FUENTE == 'roboflow':
    from google.colab import userdata
    from roboflow import Roboflow
    rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
    ds = rf.workspace(WORKSPACE).project(PROYECTO).version(VERSION).download('yolov11', location='/content/ds_pasajeros')
    DATA_YAML = f'{ds.location}/data.yaml'
else:
    from google.colab import files
    for n in files.upload():
        !unzip -qo "$n" -d /content
    DATA_YAML = glob.glob('/content/**/data.yaml', recursive=True)[0]

datos = preparar_yaml(DATA_YAML)
print('Clases del dataset:', datos['names'])
print('train:', datos['train'], '| val:', datos['val'])

## 2. Filtrar clases: personas + vehículos
Se conservan solo las clases de **persona** (renombradas a `persona`) y las de **vehículo** (renombradas a
`auto`, `bus`, `camion`, `minibus`, `moto`); el resto (semáforos, bicicletas, etc.) se elimina de las etiquetas y los
índices se reasignan. Si no hay ninguna clase de persona, la celda se detiene con un error claro.

In [ ]:
PERSONA = {'person', 'persona', 'personas', 'people', 'pasajero', 'pasajeros', 'passenger', 'passengers', 'conductor',
           'driver', 'rider', 'motorbike rider', 'motorcyclist', 'pedestrian', 'human', 'occupant', 'ocupante'}
VEHICULO = {'car': 'auto', 'auto': 'auto', 'sedan': 'auto', 'suv': 'auto', 'camioneta': 'auto', 'pickup': 'auto',
            'auto particular': 'auto', 'bus': 'bus', 'autobus': 'bus', 'truck': 'camion', 'camion': 'camion',
            'camión': 'camion', 'van': 'minibus', 'minibus': 'minibus', 'minibús': 'minibus', 'microbus': 'minibus',
            'trufi': 'minibus', 'motorcycle': 'moto', 'motorbike': 'moto', 'moto': 'moto', 'motocicleta': 'moto'}

def nueva_clase(nombre):
    n = str(nombre).strip().lower().replace('_', ' ').replace('-', ' ')
    return 'persona' if n in PERSONA else VEHICULO.get(n)

nombres = datos['names'] if isinstance(datos['names'], list) else [datos['names'][k] for k in sorted(datos['names'])]
destino = [nueva_clase(n) for n in nombres]
assert 'persona' in destino, f'El dataset no tiene una clase de persona: {nombres}. Elige otro dataset.'
NUEVAS = ['persona'] + sorted({d for d in destino if d and d != 'persona'})
mapa = {i: NUEVAS.index(d) for i, d in enumerate(destino) if d}
print('Se conservan:', {nombres[i]: NUEVAS[j] for i, j in mapa.items()})
print('Se eliminan :', [n for i, n in enumerate(nombres) if i not in mapa])

conteo = {c: 0 for c in NUEVAS}
for split in ('train', 'val', 'test'):
    if split not in datos:
        continue
    for txt in glob.glob(os.path.join(datos['path'], datos[split].replace('images', 'labels'), '*.txt')):
        lineas = []
        for l in open(txt).read().splitlines():
            v = l.split()
            if v and int(v[0]) in mapa:
                lineas.append(' '.join([str(mapa[int(v[0])])] + v[1:]))
                conteo[NUEVAS[mapa[int(v[0])]]] += 1
        open(txt, 'w').write('\n'.join(lineas))
for cache in glob.glob(os.path.join(datos['path'], '**', '*.cache'), recursive=True):
    os.remove(cache)   # las etiquetas cambiaron: se regenera el cache
datos['names'] = NUEVAS
datos['nc'] = len(NUEVAS)
yaml.safe_dump(datos, open(DATA_YAML, 'w'), allow_unicode=True, sort_keys=False)
print('Instancias por clase:', conteo)

## 3. Entrenamiento
Se parte de `yolo11s-seg.pt` (COCO ya conoce `person`). `imgsz=640` = `IMGSZ_PROPIO` del servidor.
Con poca GPU usa `yolo11n-seg.pt`.

In [ ]:
from ultralytics import YOLO
modelo = YOLO('yolo11s-seg.pt')
modelo.train(data=DATA_YAML, epochs=60, imgsz=640, batch=16, patience=20, device=0,
             fliplr=0.5, hsv_v=0.3, mosaic=1.0, close_mosaic=10,
             project='/content/runs/seg', name='pasajeros', exist_ok=True)
BEST = '/content/runs/seg/pasajeros/weights/best.pt'

## 4. Evaluación (mAP de cajas y máscaras, por clase)

In [ ]:
import pandas as pd
from IPython.display import Image, display
m = YOLO(BEST).val(data=DATA_YAML, split='val', plots=True)
print(f"mAP50 caja: {m.box.map50:.3f} | mAP50 máscara: {m.seg.map50:.3f} | mAP50-95 máscara: {m.seg.map:.3f}")
display(pd.DataFrame([{'clase': m.names[i], 'mAP50-95 máscara': round(float(m.seg.maps[i]), 3)} for i in range(len(m.names))]))
for f in ('confusion_matrix_normalized.png', 'MaskPR_curve.png', 'results.png'):
    ruta = sorted(glob.glob(f'/content/runs/**/{f}', recursive=True), key=os.path.getmtime)
    if ruta:
        display(Image(ruta[-1], width=700))

## 5. Conteo de pasajeros (misma regla que `inferencia/app/modelos.py`)
Vehículo principal = el de mayor área (de este modelo si segmenta vehículos; el servidor usa M2). Solo se dibujan
cajas: **no** se recortan ni guardan rostros. Opcional: `EVALUAR_CONTEO = True` con un CSV `pasajeros_reales.csv`
(`archivo,pasajeros`) para medir el error de conteo (MAE).

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt
modelo_p = YOLO(BEST)
IDX_PERSONA = [i for i, n in modelo_p.names.items() if n == 'persona']

def fraccion_dentro(a, b):
    x1, y1, x2, y2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    return max(0, x2 - x1) * max(0, y2 - y1) / max(1e-6, (a[2] - a[0]) * (a[3] - a[1]))

def contar_pasajeros(img, conf=0.35):
    r = modelo_p.predict(img, conf=conf, imgsz=640, verbose=False)[0]
    if r.boxes is None or len(r.boxes) == 0:
        return 0, None, []
    cajas, clases = r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy().astype(int)
    vehs = [(c, b) for c, b in zip(clases, cajas) if c not in IDX_PERSONA]
    if vehs:
        c_veh, veh = max(vehs, key=lambda t: (t[1][2] - t[1][0]) * (t[1][3] - t[1][1]))
        umbral = 0.3 if modelo_p.names[int(c_veh)] == 'moto' else 0.6
    else:
        veh, umbral = np.array([0, 0, img.shape[1], img.shape[0]]), 0.6
    personas = [b for c, b in zip(clases, cajas) if c in IDX_PERSONA and fraccion_dentro(b, veh) >= umbral]
    return len(personas), veh, personas

!wget -q -O prueba.jpg https://ultralytics.com/images/bus.jpg
img = cv2.imread('prueba.jpg')
k, veh, ps = contar_pasajeros(img)
vis = img.copy()
cv2.rectangle(vis, tuple(map(int, veh[:2])), tuple(map(int, veh[2:])), (0, 255, 0), 3)
for b in ps:
    cv2.rectangle(vis, tuple(map(int, b[:2])), tuple(map(int, b[2:])), (255, 255, 0), 2)
plt.figure(figsize=(9, 6)); plt.imshow(vis[:, :, ::-1]); plt.title(f'{k} pasajeros'); plt.axis('off'); plt.show()

EVALUAR_CONTEO = False
if EVALUAR_CONTEO:
    from google.colab import files
    for n in files.upload():
        if n.endswith('.zip'):
            !unzip -qo "$n" -d /content/eval_pasajeros
    gt = pd.read_csv('pasajeros_reales.csv')
    rutas = [a if os.path.exists(a) else os.path.join('/content/eval_pasajeros', a) for a in gt['archivo']]
    gt['predicho'] = [contar_pasajeros(cv2.imread(r))[0] for r in rutas]
    print('MAE:', round((gt['predicho'] - gt['pasajeros']).abs().mean(), 2),
          '| exactos:', round((gt['predicho'] == gt['pasajeros']).mean() * 100, 1), '%')
    display(gt)

## 6. Verificar contrato y exportar `pasajeros_seg.pt`
Comprueba lo mismo que el servidor al arrancar (`inferencia/app/validacion.py` → `validar_pasajeros`), descarga el archivo
y lo copia a Google Drive en `MyDrive/peaje/pesos/` (la carpeta que lee `colab_servidor.ipynb`).

In [ ]:
final = YOLO(BEST)
nombres_finales = [final.names[k] for k in sorted(final.names)]
assert final.task == 'segment', f'task={final.task}: debe ser segment'
assert any(str(n).strip().lower() in PERSONA for n in nombres_finales), f'Sin clase de persona: {nombres_finales}'
print('OK contrato M5 · task=segment · clases =', nombres_finales)

shutil.copy(BEST, f'/content/{NOMBRE_FINAL}')
GUARDAR_DRIVE = True
if GUARDAR_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/peaje/pesos', exist_ok=True)
    shutil.copy(f'/content/{NOMBRE_FINAL}', f'/content/drive/MyDrive/peaje/pesos/{NOMBRE_FINAL}')
    print('Guardado en Drive: MyDrive/peaje/pesos/' + NOMBRE_FINAL)
from google.colab import files
files.download(f'/content/{NOMBRE_FINAL}')